In [79]:
import pandas as pd
import os

flights = pd.read_csv('../data/raw/flights.csv')
airlines = pd.read_csv('../data/raw/airlines.csv')
airports = pd.read_csv('../data/raw/airports.csv')

C:\Users\gerar\AppData\Local\Temp\ipykernel_3404\1336779955.py:4: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv('../data/raw/flights.csv')


In [ ]:
flights.head()
flights.info()


In [ ]:
flights.shape
flights.columns
flights.isnull().sum().sort_values(ascending=False)

In [ ]:
flights[flights["ARRIVAL_DELAY"].isnull()][["CANCELLED", "DIVERTED"]].value_counts()

In [ ]:
def classify_flight(row):
    if row["CANCELLED"] == 1:
        return "CANCELLED"
    elif row["DIVERTED"] == 1:
        return "DIVERTED"
    elif row["ARRIVAL_DELAY"] >= 15:
        return "DELAYED"
    else:
        return "ON TIME / EARLY"

In [ ]:
flights_clean_all = flights.copy()
flights_clean_all["FLIGHT_STATUS"] = flights_clean_all.apply(classify_flight, axis =1)

In [ ]:
delay_causes = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

flights_clean_all[delay_causes] = flights_clean_all[delay_causes].fillna(0)

flights_clean_all["CANCELLATION_REASON"] = flights_clean_all["CANCELLATION_REASON"].fillna("Not cancelled")

In [ ]:
flights_arrived = flights_clean_all[(flights_clean_all["CANCELLED"] == 0) & (flights_clean_all["DIVERTED"] == 0)].copy()
flights_arrived.drop(columns=["CANCELLATION_REASON",])

flights_arrived.isnull().sum().sort_values(ascending=False)

In [ ]:
flights_cancelled = flights_clean_all[flights_clean_all["CANCELLED"] == 1].copy()

# Como el vuelo es cancelado, elimino las columnas que dependen de la llegada del vuelo, ya que no aplica.
cancelled_drop_cols = [
    "DEPARTURE_TIME",
    "DEPARTURE_DELAY",
    "TAXI_OUT",
    "WHEELS_OFF",
    "ELAPSED_TIME",
    "AIR_TIME",
    "WHEELS_ON",
    "TAXI_IN",
    "ARRIVAL_TIME",
    "ARRIVAL_DELAY",
    "DIVERTED",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

flights_cancelled = flights_cancelled.drop(columns=cancelled_drop_cols)
flights_cancelled.isnull().sum().sort_values(ascending=False)

In [ ]:
flights_diverted = flights_clean_all[flights_clean_all["DIVERTED"] == 1].copy()

diverted_drop_cols = [
    "ELAPSED_TIME",
    "AIR_TIME",
    "ARRIVAL_DELAY",
    "TAXI_IN",
    "WHEELS_ON",
    "ARRIVAL_TIME",
    "CANCELLED",
    "CANCELLATION_REASON",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

flights_diverted = flights_diverted.drop(columns=diverted_drop_cols)
flights_diverted.isnull().sum().sort_values(ascending=False)

In [74]:
airports[airports["IATA_CODE"].duplicated(keep=False)]

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE


In [75]:
airlines[airlines["IATA_CODE"].duplicated(keep=False)]

,IATA_CODE,AIRLINE


In [78]:
# 1. Tamaños finales
print("All:", flights_clean_all.shape)
print("Arrived:", flights_arrived.shape)
print("Cancelled:", flights_cancelled.shape)
print("Diverted:", flights_diverted.shape)
print("Airlines:", airlines.shape)
print("Airports:", airports.shape)

# 2. Validar que los estados suman el total
print(flights_clean_all["FLIGHT_STATUS"].value_counts())

# 3. Validar llaves de relación
print(airlines["IATA_CODE"].duplicated().sum())
print(airports["IATA_CODE"].duplicated().sum())

flights_clean_all[
    (flights_clean_all["FLIGHT_STATUS"] == "DELAYED") &
    (
        (flights_clean_all["CANCELLED"] == 1) |
        (flights_clean_all["DIVERTED"] == 1)
    )
].shape

All: (5819079, 32)
Arrived: (5714008, 32)
Cancelled: (89884, 16)
Diverted: (15187, 19)
Airlines: (14, 2)
Airports: (322, 7)
FLIGHT_STATUS
ON TIME / EARLY    4650569
DELAYED            1063439
CANCELLED            89884
DIVERTED             15187
Name: count, dtype: int64
0
0


(0, 32)

In [80]:
flights_clean_all.to_csv("../data/processed/flights_clean_all.csv", index=False)
flights_arrived.to_csv("../data/processed/flights_arrived.csv", index=False)
flights_cancelled.to_csv("../data/processed/flights_cancelled.csv", index=False)
flights_diverted.to_csv("../data/processed/flights_diverted.csv", index=False)

airlines.to_csv("../data/processed/airlines_clean.csv", index=False)
airports.to_csv("../data/processed/airports_clean.csv", index=False)

os.listdir("../data/processed")

['airlines_clean.csv',
 'airports_clean.csv',
 'flights_arrived.csv',
 'flights_cancelled.csv',
 'flights_clean_all.csv',
 'flights_diverted.csv']